# 03 -- LSTM: Pretrained Embeddings + Attention + Hyperparameter Search (Step 3, your part)Loads `cleaned_bfrd.csv` from Drive, downloads fastText Bangla vectors (~840MB, first run only -- saved to Drive after), trains an LSTM with attention, using cross-entropy loss and a small hyperparameter search.**Your teammate's RNN goes in its own separate notebook** -- see the note at the bottom of this one for what to keep consistent.

In [1]:
!pip install -q gensim torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 42.9 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/BanglaFakeReviewDetector'
os.chdir(PROJECT_DIR)

import pandas as pd
df = pd.read_csv("cleaned_bfrd.csv", encoding="utf-8-sig")
counts = df["label"].value_counts()
AUTHENTIC_LABEL, FAKE_LABEL = counts.idxmax(), counts.idxmin()
print(f"Loaded {len(df)} rows. authentic={AUTHENTIC_LABEL!r}, fake={FAKE_LABEL!r}")

Mounted at /content/drive
Loaded 9032 rows. authentic=np.int64(1), fake=np.int64(0)


In [3]:
import re
import nltk
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

BANGLA_STOPWORDS = set(stopwords.words('bengali'))
_porter = PorterStemmer()

EMOJI_PATTERN = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF" "\U00002190-\U000021FF" "]+", flags=re.UNICODE)

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = EMOJI_PATTERN.sub(" ", text)
    text = re.sub(r"([\u0964!?.\-_=~*])\1{1,}", r"\1", text)
    text = re.sub(r"(#\w+)(\s*\1){1,}", r"\1", text)
    return re.sub(r"\s+", " ", text).strip()

def is_bangla_token(token):
    if not token:
        return False
    bangla_chars = sum(1 for ch in token if "\u0980" <= ch <= "\u09FF")
    return bangla_chars > len(token) / 2

def basic_tokenize(text):
    return re.findall(r"[\u0980-\u09FFa-zA-Z0-9]+", text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in BANGLA_STOPWORDS]

def stem_latin_tokens(tokens):
    return [_porter.stem(t) if not is_bangla_token(t) else t for t in tokens]

def preprocess_review(raw_text):
    cleaned = clean_text(raw_text)
    tokens = basic_tokenize(cleaned)
    return {"cleaned_text": cleaned, "tokens": stem_latin_tokens(remove_stopwords(tokens))}

print(f"Loaded {len(BANGLA_STOPWORDS)} Bengali stopwords.")

Loaded 398 Bengali stopwords.


## Load pretrained Bangla embeddings (cached to Drive after first download)

In [4]:
import urllib.request, os
from gensim.models import KeyedVectors

FASTTEXT_URL = "https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.bn.300.vec.gz"
FASTTEXT_PATH = "cc.bn.300.vec.gz"  # saved inside PROJECT_DIR on Drive -- persists across sessions

if not os.path.exists(FASTTEXT_PATH):
    print("Downloading fastText Bangla vectors (~840MB, one-time -- saved to Drive after)...")
    urllib.request.urlretrieve(FASTTEXT_URL, FASTTEXT_PATH)
    print("Done.")
else:
    print("Found cached vectors on Drive -- skipping download.")

kv = KeyedVectors.load_word2vec_format(FASTTEXT_PATH, binary=False, limit=200_000)
EMBED_DIM = kv.vector_size
print(f"Loaded {len(kv)} pretrained vectors, dim={EMBED_DIM}")

Found cached vectors on Drive -- skipping download.
Loaded 200000 pretrained vectors, dim=300


## Build vocabulary, embedding matrix, and the LSTM+Attention model

In [5]:
import random
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

RANDOM_STATE = 42
random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); torch.manual_seed(RANDOM_STATE)

MAX_VOCAB_SIZE, MAX_SEQ_LEN = 8000, 40
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"

def build_vocab(tokenized_texts, max_size=MAX_VOCAB_SIZE):
    counts = Counter(tok for tokens in tokenized_texts for tok in tokens)
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for tok, _ in counts.most_common(max_size - 2):
        vocab[tok] = len(vocab)
    return vocab

def encode(tokens, vocab, max_len=MAX_SEQ_LEN):
    ids = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens[:max_len]]
    return ids + [vocab[PAD_TOKEN]] * (max_len - len(ids))

def build_embedding_matrix(vocab, kv, embed_dim):
    matrix = np.random.normal(scale=0.1, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[0] = np.zeros(embed_dim, dtype=np.float32)
    found = 0
    for word, idx in vocab.items():
        if word in kv:
            matrix[idx] = kv[word]
            found += 1
    print(f"Found pretrained vectors for {found}/{len(vocab)} words ({100*found/len(vocab):.1f}%).")
    return matrix

class LSTMAttentionClassifier(nn.Module):
    """LSTM with pretrained embeddings + additive attention over every timestep."""
    def __init__(self, embedding_matrix, hidden_dim=64, freeze_embeddings=False):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32), freeze=freeze_embeddings, padding_idx=0)
        embed_dim = embedding_matrix.shape[1]
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.attention = nn.Linear(hidden_dim, 1)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        lstm_out, _ = self.lstm(embedded)
        attn_scores = self.attention(lstm_out).squeeze(-1)
        mask = (x != 0)
        attn_scores = attn_scores.masked_fill(~mask, float("-inf"))
        attn_weights = F.softmax(attn_scores, dim=1)
        context = torch.sum(lstm_out * attn_weights.unsqueeze(-1), dim=1)
        return self.fc(context).squeeze(-1), attn_weights

print("Model class ready.")

Model class ready.


## Prepare data, run hyperparameter search, train final model

In [6]:
tokenized_all = [basic_tokenize(clean_text(t)) for t in df["text"]]
binary_labels = (df["label"] == FAKE_LABEL).astype(int).tolist()

train_tok, temp_tok, y_train_b, y_temp_b = train_test_split(
    tokenized_all, binary_labels, test_size=0.3, random_state=RANDOM_STATE, stratify=binary_labels)
val_tok, test_tok, y_val_b, y_test_b = train_test_split(
    temp_tok, y_temp_b, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp_b)

vocab = build_vocab(train_tok)
embedding_matrix = build_embedding_matrix(vocab, kv, EMBED_DIM)

X_train = [encode(t, vocab) for t in train_tok]
X_val = torch.tensor([encode(t, vocab) for t in val_tok], dtype=torch.long)
X_test = torch.tensor([encode(t, vocab) for t in test_tok], dtype=torch.long)
y_val_t = torch.tensor(y_val_b, dtype=torch.float32)
y_test_t = torch.tensor(y_test_b, dtype=torch.float32)

class_counts = [y_train_b.count(0), y_train_b.count(1)]
sample_weights = [1.0 / class_counts[l] for l in y_train_b]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(y_train_b), replacement=True)
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.long), torch.tensor(y_train_b, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=16, sampler=sampler)

def train_and_eval(hidden_dim, lr, epochs):
    model = LSTMAttentionClassifier(embedding_matrix, hidden_dim=hidden_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    cross_entropy_loss = nn.BCEWithLogitsLoss()  # binary cross-entropy = cross-entropy for 2 classes
    model.train()
    for epoch in range(epochs):
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            logits, _ = model(X_batch)
            loss = cross_entropy_loss(logits, y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
    model.eval()
    with torch.no_grad():
        val_logits, val_attn = model(X_val)
        val_preds = (torch.sigmoid(val_logits) > 0.5).float().numpy()
    return model, val_attn, f1_score(y_val_t.numpy(), val_preds, average="macro", zero_division=0)

print("=== Hyperparameter search over variable parameters ===")
results = []
for cfg in [{"hidden_dim": 32, "lr": 1e-3}, {"hidden_dim": 64, "lr": 1e-3},
            {"hidden_dim": 64, "lr": 5e-4}, {"hidden_dim": 128, "lr": 5e-4}]:
    _, _, val_f1 = train_and_eval(cfg["hidden_dim"], cfg["lr"], epochs=5)
    results.append({**cfg, "val_macro_f1": val_f1})
    print(f"  hidden_dim={cfg['hidden_dim']:>4}  lr={cfg['lr']:<7}  val_macro_f1={val_f1:.3f}")

best = max(results, key=lambda r: r["val_macro_f1"])
print(f"\nBest config: {best}")

print(f"\n=== Final run: hidden_dim={best['hidden_dim']}, lr={best['lr']} ===")
final_model, test_attn, _ = train_and_eval(best["hidden_dim"], best["lr"], epochs=15)
final_model.eval()
with torch.no_grad():
    test_logits, _ = final_model(X_test)
    test_preds = (torch.sigmoid(test_logits) > 0.5).numpy()
print(classification_report(y_test_t.numpy(), test_preds, zero_division=0))

Found pretrained vectors for 6582/8000 words (82.3%).
=== Hyperparameter search over variable parameters ===
  hidden_dim=  32  lr=0.001    val_macro_f1=0.689
  hidden_dim=  64  lr=0.001    val_macro_f1=0.681
  hidden_dim=  64  lr=0.0005   val_macro_f1=0.672
  hidden_dim= 128  lr=0.0005   val_macro_f1=0.684

Best config: {'hidden_dim': 32, 'lr': 0.001, 'val_macro_f1': 0.6892663831042594}

=== Final run: hidden_dim=32, lr=0.001 ===
              precision    recall  f1-score   support

         0.0       0.90      0.95      0.92      1155
         1.0       0.55      0.38      0.45       200

    accuracy                           0.86      1355
   macro avg       0.72      0.66      0.68      1355
weighted avg       0.85      0.86      0.85      1355



## Save the trained model

Saved the same way as the RNN notebook (`saved_lstm_model/`), so the interface notebook can load either model by name without retraining.

In [7]:
import os, joblib

os.makedirs("saved_lstm_model", exist_ok=True)
torch.save(final_model.state_dict(), "saved_lstm_model/lstm_model.pt")
joblib.dump(vocab, "saved_lstm_model/vocab.joblib")
joblib.dump({"hidden_dim": best["hidden_dim"], "embed_dim": EMBED_DIM, "max_seq_len": MAX_SEQ_LEN},
            "saved_lstm_model/config.joblib")
joblib.dump({"authentic": AUTHENTIC_LABEL, "fake": FAKE_LABEL}, "saved_lstm_model/label_map.joblib")
print("Saved to Drive -- saved_lstm_model/ -- reusable without retraining.")

Saved to Drive -- saved_lstm_model/ -- reusable without retraining.


## Note for your teammate's RNN notebookSo the final comparison is fair, their notebook should:- Load the same `cleaned_bfrd.csv` from `/content/drive/MyDrive/BanglaFakeReviewDetector/`- Use `random_state=42` for the identical train/val/test split- Report the same metrics (accuracy, precision, recall, F1, macro-F1)